# Введение в MapReduce модель на Python

In [77]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [78]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [79]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [80]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [81]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [82]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [83]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [84]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [85]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [86]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [87]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [88]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [89]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [90]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [91]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.7382842098506064)),
 (1, np.float64(2.7382842098506064)),
 (2, np.float64(2.7382842098506064)),
 (3, np.float64(2.7382842098506064)),
 (4, np.float64(2.7382842098506064))]

## Inverted index

In [92]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [93]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [94]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [95]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('banana', 2), ('it', 18)]),
 (1, [('is', 18), ('what', 10)])]

## TeraSort

In [96]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.008610513014811638)),
   (None, np.float64(0.01444391035481607)),
   (None, np.float64(0.053958088682512395)),
   (None, np.float64(0.08067029800795056)),
   (None, np.float64(0.09583373526039851)),
   (None, np.float64(0.1163808711819242)),
   (None, np.float64(0.12690018609703524)),
   (None, np.float64(0.1518488351980838)),
   (None, np.float64(0.21894336311138185)),
   (None, np.float64(0.27167700729988353)),
   (None, np.float64(0.2974335707948438)),
   (None, np.float64(0.3008712510528556)),
   (None, np.float64(0.31596074695554)),
   (None, np.float64(0.3590910573864178)),
   (None, np.float64(0.36668238130869635)),
   (None, np.float64(0.38758196656389876)),
   (None, np.float64(0.4145909331670652)),
   (None, np.float64(0.43654427614591607)),
   (None, np.float64(0.4607881555714668))]),
 (1,
  [(None, np.float64(0.5438519598947641)),
   (None, np.float64(0.5839737385396203)),
   (None, np.float64(0.7180273188286594)),
   (None, np.float64(0.72055885

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [97]:
from typing import Iterator

input_numbers = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]

def RECORDREADER():
    for (i, num) in enumerate(input_numbers):
        yield (i, num)

def MAP(index: int, value: int):
    yield ("max", value)

def REDUCE(key: str, values: Iterator[int]):
    yield (key, max(values))

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(input_numbers)
print(output)

[3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]
[('max', 9)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [98]:
from typing import Iterator

input_numbers = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]

def RECORDREADER():
    for (i, num) in enumerate(input_numbers):
        yield (i, num)

def MAP(index: int, value: int):
    yield ("mean", (value, 1))

def REDUCE(key: str, pairs: Iterator[tuple]):
    total, count = 0, 0
    for (value, n) in pairs:
        total += value
        count += n
    yield (key, total / count)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(input_numbers)
print(output)

[3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]
[('mean', 4.0)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [99]:
import itertools

def groupbykey_sort(iterable):
    sorted_pairs = sorted(iterable, key=lambda kv: kv[0])
    for key, group in itertools.groupby(sorted_pairs, key=lambda kv: kv[0]):
        yield (key, [v for (_, v) in group])

pairs = [("b", 2), ("a", 1), ("c", 3), ("a", 4), ("b", 5), ("a", 6)]

print("Input:", pairs)
print()
print("groupbykey (hash-based):")
for item in groupbykey(pairs):
    print(" ", item)

print()
print("groupbykey_sort (sort-based):")
for item in groupbykey_sort(pairs):
    print(" ", item)

print()
hash_result = dict(groupbykey(pairs))
sort_result = dict(groupbykey_sort(pairs))
print("Same keys:", set(hash_result.keys()) == set(sort_result.keys()))
print("Same values (sorted):", all(sorted(hash_result[k]) == sorted(sort_result[k]) for k in hash_result))

Input: [('b', 2), ('a', 1), ('c', 3), ('a', 4), ('b', 5), ('a', 6)]

groupbykey (hash-based):
  ('b', [2, 5])
  ('a', [1, 4, 6])
  ('c', [3])

groupbykey_sort (sort-based):
  ('a', [1, 4, 6])
  ('b', [2, 5])
  ('c', [3])

Same keys: True
Same values (sorted): True


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [100]:
import numpy as np
from typing import Iterator

input_data = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5, 9, 9, 2, 1]

maps = 3
reducers = 3

def INPUTFORMAT():
    global maps
    def RECORDREADER(split):
        for (i, value) in split:
            yield (i, value)
    split_size = int(np.ceil(len(input_data) / maps))
    for i in range(0, len(input_data), split_size):
        yield RECORDREADER(list(enumerate(input_data))[i:i+split_size])

def MAP(index: int, value: int):
    yield (value, None)

def COMBINER(value: int, nones: Iterator):
    yield (value, None)

def REDUCE(value: int, nones: Iterator):
    yield (value, None)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE,
                                          COMBINER=COMBINER)
unique_values = sorted(
    value for (_, partition) in partitioned_output for (value, _) in partition
)
print("Input:  ", input_data)
print("Unique: ", unique_values)

13 key-value pairs were sent over a network.
Input:   [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5, 9, 9, 2, 1]
Unique:  [1, 2, 3, 4, 5, 6, 9]


# Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [101]:
from typing import NamedTuple, Iterator

class Employee(NamedTuple):
    id: int
    name: str
    department: str
    salary: int

relation_R = [
    Employee(1, "Alice", "Engineering", 95000),
    Employee(2, "Bob", "Marketing", 60000),
    Employee(3, "Carol", "Engineering", 82000),
    Employee(4, "Dave", "HR", 55000),
    Employee(5, "Eve", "Engineering", 110000),
]

def C(t: Employee) -> bool:
    return t.department == "Engineering" and t.salary > 85000

def RECORDREADER():
    for t in relation_R:
        yield (t, t)

def MAP(key: Employee, t: Employee):
    if C(t):
        yield (t, t)

def REDUCE(t: Employee, values: Iterator[Employee]):
    for v in values:
        yield (t, v)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))

for i in relation_R:
    print(i)
print()

print("sigma(department='Engineering' AND salary>85000)(R):")
for (_, t) in output:
    print(" ", t)

Employee(id=1, name='Alice', department='Engineering', salary=95000)
Employee(id=2, name='Bob', department='Marketing', salary=60000)
Employee(id=3, name='Carol', department='Engineering', salary=82000)
Employee(id=4, name='Dave', department='HR', salary=55000)
Employee(id=5, name='Eve', department='Engineering', salary=110000)

sigma(department='Engineering' AND salary>85000)(R):
  Employee(id=1, name='Alice', department='Engineering', salary=95000)
  Employee(id=5, name='Eve', department='Engineering', salary=110000)


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [118]:
from typing import NamedTuple, Iterator

class Employee(NamedTuple):
    id: int
    name: str
    department: str
    salary: int

class EmployeeProjected(NamedTuple):
    name: str
    department: str

relation_R = [
    Employee(1, "Alice", "Engineering", 95000),
    Employee(2, "Bob", "Marketing", 60000),
    Employee(3, "Carol", "Engineering", 82000),
    Employee(4, "Dave", "Engineering", 82000),
    Employee(5, "Carol", "Engineering", 110000),
]

def project(t: Employee) -> EmployeeProjected:
    return EmployeeProjected(name=t.name, department=t.department)

def RECORDREADER():
    for t in relation_R:
        yield (t, t)

def MAP(key: Employee, t: Employee):
    t_prime = project(t)
    yield (t_prime, t_prime)

def REDUCE(t_prime: EmployeeProjected, values: Iterator[EmployeeProjected]):
    yield (t_prime, values[0])

output = list(MapReduce(RECORDREADER, MAP, REDUCE))

for i in relation_R:
    print(i)
print()

print(f"pi(name, department)(R) - {len(relation_R)} rows; {len(output)} distinct rows:")
for (_, t_prime) in output:
    print(" ", t_prime)

Employee(id=1, name='Alice', department='Engineering', salary=95000)
Employee(id=2, name='Bob', department='Marketing', salary=60000)
Employee(id=3, name='Carol', department='Engineering', salary=82000)
Employee(id=4, name='Dave', department='Engineering', salary=82000)
Employee(id=5, name='Carol', department='Engineering', salary=110000)

pi(name, department)(R) - 5 rows; 4 distinct rows:
  EmployeeProjected(name='Alice', department='Engineering')
  EmployeeProjected(name='Bob', department='Marketing')
  EmployeeProjected(name='Carol', department='Engineering')
  EmployeeProjected(name='Dave', department='Engineering')


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [117]:
from typing import NamedTuple, Iterator

class Employee(NamedTuple):
    id: int
    name: str
    department: str

relation_R = [
    Employee(1, "Alice", "Engineering"),
    Employee(2, "Bob", "Marketing"),
    Employee(3, "Carol", "Engineering"),
]

relation_S = [
    Employee(3, "Carol", "Engineering"),
    Employee(4, "Dave", "HR"),
    Employee(5, "Eve", "Engineering"),
]

def RECORDREADER():
    for t in relation_R:
        yield (t, t)
    for t in relation_S:
        yield (t, t)

def MAP(key, t):
    yield (t, t)

def REDUCE(t, values: Iterator):
    yield (t, values[0])

output = list(MapReduce(RECORDREADER, MAP, REDUCE))

for i in relation_R:
    print(i)
print()

for i in relation_S:
    print(i)
print()

print(f"R union S - |R| = {len(relation_R)}, |S| = {len(relation_S)}; {len(output)} distinct tuples:")
for (_, t) in output:
    print(" ", t)

Employee(id=1, name='Alice', department='Engineering')
Employee(id=2, name='Bob', department='Marketing')
Employee(id=3, name='Carol', department='Engineering')

Employee(id=3, name='Carol', department='Engineering')
Employee(id=4, name='Dave', department='HR')
Employee(id=5, name='Eve', department='Engineering')

R union S - |R| = 3, |S| = 3; 5 distinct tuples:
  Employee(id=1, name='Alice', department='Engineering')
  Employee(id=2, name='Bob', department='Marketing')
  Employee(id=3, name='Carol', department='Engineering')
  Employee(id=4, name='Dave', department='HR')
  Employee(id=5, name='Eve', department='Engineering')


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [ ]:
from typing import NamedTuple, Iterator

class Employee(NamedTuple):
    id: int
    name: str
    department: str

relation_R = [
    Employee(1, "Alice", "Engineering"),
    Employee(2, "Bob", "Marketing"),
    Employee(3, "Carol", "Engineering"),
]

relation_S = [
    Employee(3, "Carol", "Engineering"),
    Employee(4, "Dave", "HR"),
    Employee(5, "Eve", "Engineering"),
]

def RECORDREADER():
    for t in relation_R:
        yield (t, t)
    for t in relation_S:
        yield (t, t)

def MAP(key, t):
    yield (t, t)

def REDUCE(t, values: Iterator):
    copies = list(values)
    if len(copies) == 2:
        yield (t, t)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))

for i in relation_R:
    print(i)
print()

for i in relation_S:
    print(i)
print()

print(f"R intersection S - |R| = {len(relation_R)}, |S| = {len(relation_S)}; {len(output)} tuples:")
for (_, t) in output:
    print(" ", t)

Employee(id=1, name='Alice', department='Engineering')
Employee(id=2, name='Bob', department='Marketing')
Employee(id=3, name='Carol', department='Engineering')

Employee(id=3, name='Carol', department='Engineering')
Employee(id=4, name='Dave', department='HR')
Employee(id=5, name='Eve', department='Engineering')

R intersection S - |R|=3, |S|=3 - 1 tuples:
  Employee(id=3, name='Carol', department='Engineering')


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [119]:
from typing import NamedTuple, Iterator

class Employee(NamedTuple):
    id: int
    name: str
    department: str

relation_R = [
    Employee(1, "Alice", "Engineering"),
    Employee(2, "Bob", "Marketing"),
    Employee(3, "Carol", "Engineering"),
]

relation_S = [
    Employee(3, "Carol", "Engineering"),
    Employee(4, "Dave", "HR"),
    Employee(5, "Eve", "Engineering"),
]

R, S = "R", "S"

def RECORDREADER():
    for t in relation_R:
        yield (t, R)
    for t in relation_S:
        yield (t, S)

def MAP(t, relation_tag: str):
    yield (t, relation_tag)

def REDUCE(t, tags: Iterator[str]):
    seen = set(tags)
    if seen == {R}:
        yield (t, t)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))

for i in relation_R:
    print(i)
print()

for i in relation_S:
    print(i)
print()

print(f"R - S - |R| = {len(relation_R)}, |S| = {len(relation_S)}; {len(output)} tuples:")
for (_, t) in output:
    print(" ", t)

Employee(id=1, name='Alice', department='Engineering')
Employee(id=2, name='Bob', department='Marketing')
Employee(id=3, name='Carol', department='Engineering')

Employee(id=3, name='Carol', department='Engineering')
Employee(id=4, name='Dave', department='HR')
Employee(id=5, name='Eve', department='Engineering')

R - S - |R| = 3, |S| = 3; 2 tuples:
  Employee(id=1, name='Alice', department='Engineering')
  Employee(id=2, name='Bob', department='Marketing')


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [121]:
from typing import Iterator

relation_R = [
    ("Alice", 10),
    ("Bob", 20),
    ("Carol", 10),
    ("Dave", 30),
]

relation_S = [
    (10, "Engineering"),
    (20, "Marketing"),
]

R, S = "R", "S"

def RECORDREADER():
    for (a, b) in relation_R:
        yield (b, (R, a))
    for (b, c) in relation_S:
        yield (b, (S, c))

def MAP(b, tagged_value):
    yield (b, tagged_value)

def REDUCE(b, tagged_values: Iterator):
    from_R, from_S = [], []
    for (relation_tag, val) in tagged_values:
        if relation_tag == R:
            from_R.append(val)
        else:
            from_S.append(val)
    for a in from_R:
        for c in from_S:
            yield (None, (a, b, c))

output = list(MapReduce(RECORDREADER, MAP, REDUCE))

for i in relation_R:
    print(i)
print()

for i in relation_S:
    print(i)
print()

print("R natural join S on department id:")
for (_, (a, b, c)) in output:
    print(f"  name = {a}, dept_id = {b}, dept = {c}")

('Alice', 10)
('Bob', 20)
('Carol', 10)
('Dave', 30)

(10, 'Engineering')
(20, 'Marketing')

R natural join S on department id:
  name = Alice, dept_id = 10, dept = Engineering
  name = Carol, dept_id = 10, dept = Engineering
  name = Bob, dept_id = 20, dept = Marketing


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [107]:
from typing import NamedTuple, Iterator, Callable

class Employee(NamedTuple):
    name: str
    department: str
    salary: int

relation_R = [
    Employee("Alice", "Engineering", 95000),
    Employee("Bob", "Marketing", 60000),
    Employee("Carol", "Engineering", 82000),
    Employee("Dave", "Marketing", 72000),
    Employee("Eve", "Engineering", 110000),
    Employee("Frank", "HR", 55000),
]

def RECORDREADER():
    for t in relation_R:
        yield (t, t)

def MAP(key, t: Employee):
    a, b = t.department, t.salary
    yield (a, b)

def make_reduce(theta: Callable) -> Callable:
    def REDUCE(a: str, values: Iterator[int]):
        yield (a, theta(list(values)))
    return REDUCE

aggregations = {
    "SUM": sum,
    "MAX": max,
    "MIN": min,
    "COUNT": len,
    "AVG": lambda vs: sum(vs) / len(vs),
}

for i in relation_R:
    print(i)
print()

for name, theta in aggregations.items():
    output = sorted(MapReduce(RECORDREADER, MAP, make_reduce(theta)))
    print(f"gamma(department, {name}(salary)):")
    for (dept, result) in output:
        print(f"  {dept:15} {result}")
    print()

Employee(name='Alice', department='Engineering', salary=95000)
Employee(name='Bob', department='Marketing', salary=60000)
Employee(name='Carol', department='Engineering', salary=82000)
Employee(name='Dave', department='Marketing', salary=72000)
Employee(name='Eve', department='Engineering', salary=110000)
Employee(name='Frank', department='HR', salary=55000)

gamma(department, SUM(salary)):
  Engineering     287000
  HR              55000
  Marketing       132000

gamma(department, MAX(salary)):
  Engineering     110000
  HR              55000
  Marketing       72000

gamma(department, MIN(salary)):
  Engineering     82000
  HR              55000
  Marketing       60000

gamma(department, COUNT(salary)):
  Engineering     3
  HR              1
  Marketing       2

gamma(department, AVG(salary)):
  Engineering     95666.66666666667
  HR              55000.0
  Marketing       66000.0



#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [122]:
import numpy as np

I, J = 4, 6
mat = np.random.rand(I, J)
vec = np.random.rand(J)

def RECORDREADER():
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            yield (("M", i, j), mat[i, j])
    for j in range(len(vec)):
        yield (("v", j), vec[j])

def MAP(key, value):
    if key[0] == "M":
        _, i, j = key
        yield (j, ("M", i, value))
    else:
        _, j = key
        yield (j, ("v", value))

def REDUCE(j, tagged_values):
    v_j = None
    m_entries = []
    for tag in tagged_values:
        if tag[0] == "M":
            m_entries.append((tag[1], tag[2]))
        else:
            v_j = tag[1]
    if v_j is not None:
        for (i, m_ij) in m_entries:
            yield (i, m_ij * v_j)

pass1 = list(MapReduce(RECORDREADER, MAP, REDUCE))

def RECORDREADER2():
    return pass1

def MAP2(i, partial_product):
    yield (i, partial_product)

def REDUCE2(i, partials):
    yield (i, sum(partials))

result = sorted(MapReduce(RECORDREADER2, MAP2, REDUCE2))

reference = np.matmul(mat, vec)
computed  = np.array([v for (_, v) in result])

print(mat)
print()

print(vec)
print()

print("Result matches numpy:", np.allclose(reference, computed))
print("M * v =", computed)

[[0.8271607  0.9841371  0.08184803 0.35402492 0.41037764 0.73366786]
 [0.93250733 0.68659443 0.00498296 0.02442311 0.64207429 0.86212539]
 [0.43663707 0.02313731 0.17519343 0.47975698 0.83567835 0.01731154]
 [0.6972962  0.34690492 0.84164759 0.6777249  0.42393344 0.63983565]]

[0.76023653 0.85548464 0.28591653 0.28764758 0.35651486 0.15505887]

Result matches numpy: True
M * v = [1.85605551 1.6673363  0.84044876 1.51281953]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [123]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [124]:
import numpy as np

I, J, K = 2, 3, 40
small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j, k), big_mat[j, k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(small_mat.shape[0]):
    yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
  (i, k) = key
  yield ((i, k), sum(values))

Проверьте своё решение

In [125]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

print(small_mat)
print()
print(big_mat)
np.allclose(reference_solution, asmatrix(solution)) # should return true

[[0.62308153 0.67506728 0.42269999]
 [0.23468229 0.11364259 0.07811662]]

[[0.48322772 0.24949396 0.8756805  0.8415189  0.77242175 0.35554573
  0.64387047 0.97523745 0.60060442 0.87226279 0.62910545 0.9027781
  0.24519664 0.10848789 0.83919289 0.77703334 0.9920351  0.82960354
  0.32452788 0.36191668 0.64594351 0.02574157 0.49102943 0.63311881
  0.23222834 0.31972127 0.12692781 0.18161582 0.06760264 0.7231174
  0.24821548 0.3453272  0.43749719 0.17607619 0.21967671 0.79780681
  0.68100797 0.11751602 0.40515492 0.47138582]
 [0.2953403  0.44483321 0.33776775 0.43001194 0.08228497 0.38779309
  0.84192757 0.6642175  0.55719408 0.84592816 0.64641925 0.89366647
  0.01769176 0.23801236 0.05671346 0.60728153 0.57780732 0.62180023
  0.37057816 0.67148872 0.76062245 0.93948503 0.4519459  0.44876985
  0.66065414 0.70971818 0.27320307 0.60414094 0.21419081 0.90245134
  0.38926467 0.84922526 0.18430352 0.12970492 0.42147254 0.03537043
  0.08653559 0.96817302 0.40899374 0.44349204]
 [0.24298612 0.634

True

In [112]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i, k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [113]:
import numpy as np

I, J, K = 3, 4, 5
mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)

def RECORDREADER():
    for i in range(mat_M.shape[0]):
        for j in range(mat_M.shape[1]):
            yield (j, ("M", i, mat_M[i, j]))
    for j in range(mat_N.shape[0]):
        for k in range(mat_N.shape[1]):
            yield (j, ("N", k, mat_N[j, k]))

def MAP(j, tagged_value):
    yield (j, tagged_value)

def REDUCE(j, tagged_values):
    m_entries, n_entries = [], []
    for tag in tagged_values:
        if tag[0] == "M":
            m_entries.append((tag[1], tag[2]))
        else:
            n_entries.append((tag[1], tag[2]))
    for (i, m_ij) in m_entries:
        for (k, n_jk) in n_entries:
            yield ((i, k), m_ij * n_jk)

pass1 = list(MapReduce(RECORDREADER, MAP, REDUCE))

def RECORDREADER2():
    return pass1

def MAP2(ik, partial_product):
    yield (ik, partial_product)

def REDUCE2(ik, partials):
    yield (ik, sum(partials))

result = sorted(MapReduce(RECORDREADER2, MAP2, REDUCE2))

reference = np.matmul(mat_M, mat_N)

def as_matrix(output, shape):
    mat = np.empty(shape)
    for ((i, k), v) in output:
        mat[i, k] = v
    return mat

print(mat_M)
print()
print(mat_N)
print()
print("Result matches numpy:", np.allclose(reference, as_matrix(result, (I, K))))

[[0.47521069 0.79435943 0.38528753 0.0376204 ]
 [0.2087354  0.33527196 0.62680546 0.52140215]
 [0.04464254 0.73858522 0.63276972 0.14678667]]

[[0.02329545 0.38896416 0.51089207 0.71143639 0.57234609]
 [0.83359721 0.0482689  0.43747987 0.71012271 0.79015892]
 [0.00667093 0.32744585 0.40677679 0.80356714 0.58549545]
 [0.92618285 0.67057106 0.37882405 0.580912   0.66400134]]

Result matches numpy: True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [114]:
import numpy as np

I, J, K = 3, 4, 5
mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)
reducers = 3

def INPUTFORMAT():
    def RECORDREADER_M():
        for i in range(mat_M.shape[0]):
            for j in range(mat_M.shape[1]):
                yield (j, ("M", i, mat_M[i, j]))

    def RECORDREADER_N():
        for j in range(mat_N.shape[0]):
            for k in range(mat_N.shape[1]):
                yield (j, ("N", k, mat_N[j, k]))

    yield RECORDREADER_M()
    yield RECORDREADER_N()

def MAP(j, tagged_value):
    yield (j, tagged_value)

def REDUCE(j, tagged_values):
    m_entries, n_entries = [], []
    for tag in tagged_values:
        if tag[0] == "M":
            m_entries.append((tag[1], tag[2]))
        else:
            n_entries.append((tag[1], tag[2]))
    for (i, m_ij) in m_entries:
        for (k, n_jk) in n_entries:
            yield ((i, k), m_ij * n_jk)

pass1_partitioned = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)
pass1 = [(ik, v) for (_, partition) in pass1_partitioned for (ik, v) in partition]

def INPUTFORMAT2():
    split_size = max(1, len(pass1) // reducers)
    def RECORDREADER2(split):
        return iter(split)
    for i in range(0, len(pass1), split_size):
        yield RECORDREADER2(pass1[i:i+split_size])

def MAP2(ik, partial):
    yield (ik, partial)

def REDUCE2(ik, partials):
    yield (ik, sum(partials))

pass2_partitioned = MapReduceDistributed(INPUTFORMAT2, MAP2, REDUCE2)
result = sorted(ik_v for (_, partition) in pass2_partitioned for ik_v in partition)

reference = np.matmul(mat_M, mat_N)

def as_matrix(output, shape):
    mat = np.empty(shape)
    for ((i, k), v) in output:
        mat[i, k] = v
    return mat

print(mat_M)
print()
print(mat_N)
print()
print("Result matches numpy:", np.allclose(reference, as_matrix(result, (I, K))))

32 key-value pairs were sent over a network.
60 key-value pairs were sent over a network.
[[0.36190872 0.92305254 0.57749267 0.27872451]
 [0.73200059 0.1904664  0.83332004 0.78964648]
 [0.20836218 0.6162226  0.77256745 0.61335928]]

[[0.39216338 0.66941139 0.00739833 0.96442145 0.61850972]
 [0.02857762 0.32398417 0.28233718 0.76262793 0.18039798]
 [0.0964571  0.43573109 0.68558984 0.56867973 0.18164154]
 [0.9017078  0.23906852 0.88266593 0.23027086 0.4161664 ]]

Result matches numpy: True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [127]:
import numpy as np

I, J, K = 4, 5, 6
mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)
maps_M = 2
maps_N = 3
reducers = 3

def run_matmul(elements_M, elements_N):
    def INPUTFORMAT():
        m_split = int(np.ceil(len(elements_M) / maps_M))
        for start in range(0, len(elements_M), m_split):
            chunk = elements_M[start:start+m_split]
            def reader_M(c=chunk):
                for (i, j, v) in c:
                    yield (j, ("M", i, v))
            yield reader_M()

        n_split = int(np.ceil(len(elements_N) / maps_N))
        for start in range(0, len(elements_N), n_split):
            chunk = elements_N[start:start+n_split]
            def reader_N(c=chunk):
                for (j, k, v) in c:
                    yield (j, ("N", k, v))
            yield reader_N()

    def MAP(j, tagged):
        yield (j, tagged)

    def REDUCE(j, tagged_values):
        m_entries, n_entries = [], []
        for tag in tagged_values:
            if tag[0] == "M":
                m_entries.append((tag[1], tag[2]))
            else:
                n_entries.append((tag[1], tag[2]))
        for (i, m_ij) in m_entries:
            for (k, n_jk) in n_entries:
                yield ((i, k), m_ij * n_jk)

    pass1 = [(ik, v) for (_, part) in MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)
                      for (ik, v) in part]

    def INPUTFORMAT2():
        split = int(np.ceil(len(pass1) / reducers)) if pass1 else 1
        for start in range(0, max(len(pass1), 1), split):
            chunk = pass1[start:start+split]
            yield iter(chunk)

    def MAP2(ik, partial):
        yield (ik, partial)

    def REDUCE2(ik, partials):
        yield (ik, sum(partials))

    result = sorted(ik_v for (_, part) in MapReduceDistributed(INPUTFORMAT2, MAP2, REDUCE2)
                         for ik_v in part)
    out = np.zeros((I, K))
    for ((i, k), v) in result:
        out[i, k] = v
    return out

all_M = [(i, j, mat_M[i, j]) for i in range(I) for j in range(J)]
all_N = [(j, k, mat_N[j, k]) for j in range(J) for k in range(K)]

result_full = run_matmul(all_M, all_N)
reference = np.matmul(mat_M, mat_N)
print()
print(f"Test 1 - full matrices, {maps_M} RECORDREADERs for M, {maps_N} for N:")
print("  Matches numpy:", np.allclose(reference, result_full))
print()

rng = np.random.default_rng(42)
keep_M = rng.choice(len(all_M), size=len(all_M)//2, replace=False)
keep_N = rng.choice(len(all_N), size=len(all_N)//2, replace=False)
subset_M = [all_M[i] for i in keep_M]
subset_N = [all_N[i] for i in keep_N]

result_sub = run_matmul(subset_M, subset_N)

expected_sub = np.zeros((I, K))
for (i, j, m_ij) in subset_M:
    for (j2, k, n_jk) in subset_N:
        if j == j2:
            expected_sub[i, k] += m_ij * n_jk

print(f"\nTest 2 - random 50% subsets of M and N:")
print("  Matches full M * N: ", np.allclose(reference, result_sub))
print("  Matches partial M * N:", np.allclose(expected_sub, result_sub))

50 key-value pairs were sent over a network.
120 key-value pairs were sent over a network.

Test 1 - full matrices, 2 RECORDREADERs for M, 3 for N:
  Matches numpy: True

25 key-value pairs were sent over a network.
28 key-value pairs were sent over a network.

Test 2 - random 50% subsets of M and N:
  Matches full M * N:  False
  Matches partial M * N: True
